### ЗАДАЧА: Пакетная загрузка конфигов деплоя

От DevOps-команды приходит пакет строк с конфигами сервисов для выкладки.
Нужно обработать их так, чтобы:
- валидные конфиги попали в итоговый список,
- проблемные записи не остановили весь пакет,
- по ошибкам собрался отдельный журнал,
- в конце было видно, какие сервисы включены по окружениям и какой у них средний timeout.

Часть строк содержит ошибки в формате и числах,
часть использует неизвестное окружение или неправильный флаг включения.


In [7]:
# service|max_retries|timeout_sec|environment|enabled
rows = [
    'auth|3|1.5|prod|on',
    'billing|0|2.0|stage|on',
    'search|two|0.8|dev|off',
    'media|5|-1|prod|on',
    'chat|4|1.2|test|off',
    'mail|2|0.5|stage|maybe',
    'worker|1|3.4|prod|on',
]

allowed_env = set(['prod', 'stage', 'dev'])
allowed_status = set(['on', 'of'])

class DeployConfigError(Exception):
    pass


class RowFormatError(DeployConfigError):
    pass


class RetriesError(DeployConfigError):
    pass


class TimeoutError(DeployConfigError):
    pass


class EnvironmentError(DeployConfigError):
    pass


class EnabledFlagError(DeployConfigError):
    pass


def parse_config(row:str):
    # TODO: распарсить строку и провалидировать max_retries, timeout_sec, environment, enabled
    # TODO: при ошибках конвертации использовать raise ... from ...
    # TODO: enabled вернуть как bool
    arr = row.split('|')
    service, max_retries, timeout_sec, environment, enabled = arr[0], arr[1], float(arr[2]), arr[3], arr[4]
    try:
        max_retries = int(max_retries)
    except ValueError as e:
        raise RetriesError('число попыток должно быть числом!') from e
    if timeout_sec < 0:
        raise TimeoutError('число попыток должно быть больше нуля!')
    if environment not in allowed_env:
        raise EnvironmentError('некорректная стадия!')
    if enabled not in allowed_status:
        raise EnabledFlagError('некорректный статус!')
    enabled = True if enabled == 'on' else False
    return {'service': service, 'max_retries': max_retries, 'timeout_sec': timeout_sec, 'environment': environment, 'enabled': enabled}
    
    
    



def load_configs(rows):
    # TODO: вернуть (configs, errors)
    configs = []
    errors = []
    for el in rows:
        try:
            obj = parse_config(el)
            configs.append(obj)
        except Exception as e:
            errors.append((el, type(e).__name__, e))
    return (configs, errors)

# TODO: вызвать load_configs(rows)
# TODO: вывести число валидных конфигов и число ошибок
# TODO: вывести ошибки по типам
# TODO: собрать enabled_by_environment: dict[str, list[str]]
# TODO: посчитать average_timeout только по enabled=True
res = load_configs(rows)
print(f'Число валидных конфигов: {len(res[0])}')
print(f'Число ошибок: {len(res[1])}')
types_err = {}
for el in res[1]:
    types_err.setdefault(el[1], [])
    types_err[el[1]].append(el)
print(types_err)

enabled_by_env = {}
for el in res[0]:
    enabled_by_env.setdefault(el['environment'], [])
    enabled_by_env[el['environment']].append(el)

print('--------------------')
print(enabled_by_env)
arr = []
for el in res[0]:
    if el['enabled'] == True:
        arr.append(el['timeout_sec'])

print(f'average timeout: {sum(arr) / len(arr)}')



Число валидных конфигов: 3
Число ошибок: 4
{'RetriesError': [('search|two|0.8|dev|off', 'RetriesError', RetriesError('число попыток должно быть числом!'))], 'TimeoutError': [('media|5|-1|prod|on', 'TimeoutError', TimeoutError('число попыток должно быть больше нуля!'))], 'EnvironmentError': [('chat|4|1.2|test|off', 'EnvironmentError', EnvironmentError('некорректная стадия!'))], 'EnabledFlagError': [('mail|2|0.5|stage|maybe', 'EnabledFlagError', EnabledFlagError('некорректный статус!'))]}
--------------------
{'prod': [{'service': 'auth', 'max_retries': 3, 'timeout_sec': 1.5, 'environment': 'prod', 'enabled': True}, {'service': 'worker', 'max_retries': 1, 'timeout_sec': 3.4, 'environment': 'prod', 'enabled': True}], 'stage': [{'service': 'billing', 'max_retries': 0, 'timeout_sec': 2.0, 'environment': 'stage', 'enabled': True}]}
average timeout: 2.3000000000000003
